In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys,os
import math
from pathlib import Path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))
#sys.path.append(r"\Users\13347\Documents\Yale\Paraproducts\Quasilinearization-of-potential-and-Schrodinger-kernels-with-tensor-paraproducts-main")
from tensor_paraproducts import haar_paraproduct
import pickle
import time

In [2]:

# Initialization
alf = [5e-3,5e-2,5e-1]
dimlist = [128,256,512]
bsvtbl = np.zeros((3,3,3))


# Data generation
for d in range(len(dimlist)):

    N = dimlist[d]
    
    # Generate coordinates
    hill_x = np.linspace(0.4, 0.6, N)
    hill_y = np.exp(-((hill_x - 0.5) / .1) ** 2) *.6
    
    xloc = 0.61
    line_x = np.full(N, xloc)
    line_y = np.linspace(0, 0.4, N)
    
    hill_x = np.expand_dims(hill_x,axis=1)
    hill_y = np.expand_dims(hill_y,axis=1)
    line_x = np.expand_dims(line_x,axis=1)
    line_y = np.expand_dims(line_y,axis=1)
    hill = np.concatenate((hill_x,hill_y),axis=1)
    line = np.concatenate((line_x,line_y),axis=1)
    
    
    # Compute distance matrix
    dxy = np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            dxy[i,j] = np.linalg.norm(hill[i] - line[j])
    
    # Compute polynomial
    deg = 5
    poly = dxy**(-(deg+1))
    
    # Compute potential kernel
    Apot = np.log(poly + 1e-16)
    
    # Normalization
    npoly = poly/np.max(np.abs(poly))
    nApot = Apot/np.max(np.abs(Apot))

    # Initiate classs
    hp = haar_paraproduct(npoly,nApot) 
    hp.p = 1
    hp.kernel = 'conpot'
    jtwo = int(math.log2(N)) - 2
    sclst = [math.log2(N), math.log2(N) + 1, math.log2(N) + 2]
    for i in range(len(alf)):
        for j in range(len(sclst)):

            jone = int(sclst[j])
            tpaof = hp.fast_tpa(jone)
            hp.alf = alf[i]
            _, _, appbnrm = hp.besov_norm(tpaof,jtwo,jtwo)
            _, _, Afbnrm = hp.besov_norm(nApot,jtwo,jtwo)
    
            bsvtbl[i,j,d] = appbnrm/Afbnrm
    


In [3]:

print(bsvtbl[:,:,0].T)

print(bsvtbl[:,:,1].T)

print(bsvtbl[:,:,2].T)



[[0.4460792  0.43229577 0.29097223]
 [0.47442689 0.47110893 0.41690344]
 [0.48937894 0.49593045 0.56037313]]
[[0.40442811 0.39321174 0.27171399]
 [0.58803294 0.5824685  0.49911635]
 [0.62987939 0.63789242 0.71071127]]
[[0.45235493 0.43824592 0.2933982 ]
 [0.82084358 0.80948325 0.66714776]
 [0.98115009 0.98272834 0.9835038 ]]


In [4]:
N = 512
eps = 2**(-(math.log2(N)))
print(eps)
eps = 2**(-(math.log2(N)+1))
print(eps)
eps = 2**(-(math.log2(N)+2))
print(eps)


0.001953125
0.0009765625
0.00048828125
